# QuantJourney SDK - ROIC WACC and Cash Conversion Quality

This notebook demonstrates a QuantJourney SDK workflow that uses statements, ratios, key metrics, market rates and profile data to compute ROIC-WACC spread, reinvestment and cash conversion quality.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
symbols = ['MSFT', 'AAPL', 'NVDA', 'GOOGL', 'META', 'AMZN', 'AVGO', 'COST']
income_raw = {symbol: qj.fmp.get_income_statement(symbol=symbol, period='annual', limit=5) for symbol in symbols}
balance_raw = {symbol: qj.fmp.get_balance_sheet_statement(symbol=symbol, period='annual', limit=5) for symbol in symbols}
cash_flow_raw = {symbol: qj.fmp.get_cash_flow_statement(symbol=symbol, period='annual', limit=5) for symbol in symbols}
key_metrics_raw = {symbol: qj.fmp.get_key_metrics_ttm(symbol=symbol) for symbol in symbols}
ratios_raw = {symbol: qj.fmp.get_financial_ratios_ttm(symbol=symbol) for symbol in symbols}
ten_y_raw = qj.fred.get_treasury_10y(start_date='2020-01-01')
prices, volumes = price_panel(symbols, start='2021-01-01', end=END)


In [ ]:
def first_row(payload: Any) -> dict[str, Any]:
    rows = as_rows(payload)
    return rows[0] if rows and isinstance(rows[0], dict) else {}
quality_rows = []
for symbol in symbols:
    income = first_row(income_raw[symbol])
    balance = first_row(balance_raw[symbol])
    cash_flow = first_row(cash_flow_raw[symbol])
    metrics = first_row(key_metrics_raw[symbol])
    ratios = first_row(ratios_raw[symbol])
    nopat = pd.to_numeric(income.get('operatingIncome') or income.get('ebit'), errors='coerce') * 0.79
    invested_capital = pd.to_numeric(balance.get('totalDebt'), errors='coerce') + pd.to_numeric(balance.get('totalStockholdersEquity'), errors='coerce') - pd.to_numeric(balance.get('cashAndCashEquivalents'), errors='coerce')
    roic = nopat / invested_capital if invested_capital else np.nan
    fcf = pd.to_numeric(cash_flow.get('freeCashFlow'), errors='coerce')
    net_income = pd.to_numeric(income.get('netIncome'), errors='coerce')
    quality_rows.append({'symbol': symbol, 'roic': roic, 'wacc_proxy': pd.to_numeric(metrics.get('weightedAverageCostOfCapital'), errors='coerce') if metrics else np.nan, 'fcf_conversion': fcf / net_income if net_income else np.nan, 'gross_margin_ttm': pd.to_numeric(ratios.get('grossProfitMarginTTM'), errors='coerce'), 'debt_to_equity_ttm': pd.to_numeric(ratios.get('debtEquityRatioTTM'), errors='coerce')})
quality = pd.DataFrame(quality_rows).set_index('symbol')


In [ ]:
if quality['wacc_proxy'].isna().all():
    rate_rows = pd.DataFrame(as_rows(ten_y_raw))
    risk_free = pd.to_numeric(rate_rows.select_dtypes(include='number').stack(), errors='coerce').dropna().tail(1).mean() / 100
    quality['wacc_proxy'] = risk_free + 0.055
quality['roic_wacc_spread'] = quality['roic'] - quality['wacc_proxy']
quality['quality_score'] = quality['roic_wacc_spread'].rank(pct=True) + quality['fcf_conversion'].rank(pct=True) + quality['gross_margin_ttm'].rank(pct=True)
display(quality.sort_values('quality_score', ascending=False))
quality[['roic', 'wacc_proxy', 'roic_wacc_spread', 'fcf_conversion']].plot(kind='bar', subplots=True, layout=(2, 2), figsize=(14, 7), title='ROIC, WACC and cash conversion')
plt.tight_layout()
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.